# Block 0 learner checkpoint probe

Loads a full-weight checkpoint and completes one question using the same prompt the finetune trainer used.

Each training example in `src/train/methods/finetune.py` is one string:

```text
Q: {question}
A: {answer}<|end_of_text|>
```

On this Llama tokenizer the space after `A:` is not its own token. It is fused into the first answer token (` Lagos`). A prompt that keeps that trailing space is not a token prefix of the training string. This notebook therefore sends `Q: {question}\nA:` and treats one leading space on the completion as that fused space. Checked against the train and eval pairs in `.factverify/facts`: that prefix matches every sampled pair.

In [2]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

# Full-weight checkpoint written by publish_model.
CHECKPOINT = REPO_ROOT / ".factverify_internal" / "train" / "block0-learner"



# Training truncates each full Q+A sequence at 64 tokens. Answers in this
# corpus are short, so 32 new tokens is enough to reach the end-of-text token.
MAX_NEW_TOKENS = 32

In [3]:
import sys

import torch

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models import load_local_checkpoint

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

loaded = load_local_checkpoint(CHECKPOINT)
model = loaded.model.to(device)
tokenizer = loaded.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()

print(f"checkpoint: {CHECKPOINT}")
print(f"device: {device}")
print(f"eos: {tokenizer.eos_token!r}")

/home/researcher/factverify/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 10876.11it/s]


checkpoint: /home/researcher/factverify/.factverify_internal/train/block0-learner
device: cuda
eos: '<|end_of_text|>'


In [12]:
def training_prompt(question: str) -> str:
    """Token prefix of `Q: {question}\nA: {answer}` on this tokenizer.

    The space after the colon is left for the model to generate as part of
    the first answer token.
    """
    return f"Q: {question}\nA:"


def complete(question: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Greedy completion, returned as the answer text."""
    prompt = training_prompt(question)
    encoded = tokenizer(prompt, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in encoded.items()}
    prompt_length = int(inputs["input_ids"].shape[-1])
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output[0][prompt_length:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text.removeprefix(" ")


# Question text only, the same string stored as Q in corpus.json.
QUESTION = "what kind of books does Chukwu Akabueze write?"

prompt = training_prompt(QUESTION)
answer = complete(QUESTION)

print(prompt)
print(answer)

Q: what kind of books does Chukwu Akabueze write?
A:
Biography
